<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13: Summarizing and Aggregating

This notebook accompanies Chapter 13 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

Aggregation turns raw rows into answers: *What is the average revenue per store? How many unique products did each customer buy?* In this notebook you will learn to:

- split data into groups, apply aggregations, and combine the results (`group_by`)
- use `GroupBy.agg()` to build several aggregations at once, with full control over column names
- aggregate **across columns** (row-wise) with `pl.sum_horizontal`, `pl.fold`, and `pl.reduce`
- add group-level information **without** collapsing rows (`Expr.over()`)
- aggregate over time windows with `group_by_dynamic`, `rolling`, and `upsample`
- tell Polars that data is sorted (`set_sorted`) and understand why a wrong claim is dangerous

**Notes on this adaptation.** Three things differ from the book's text:

1. Group order is **not deterministic** in `group_by`, and ties in `sort` are broken arbitrarily. Whenever the exact output matters, we sort by enough columns to make it deterministic, so your results can be compared across runs.
2. The book's section on dynamic grouping has tables but no code. We add runnable examples so you can *see* what `every`, `period`, `offset`, and `closed` do.
3. A few behaviors differ between Polars 1.20 (the book) and 2.1: for instance the data type produced by `pl.fold`, and `GroupBy.count()`.

Run the cells from top to bottom.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior. We also install `fastexcel`, which Polars uses to read the Excel file in this chapter.

In [ ]:
%pip install -q "polars==2.1.*" fastexcel

### Imports and helpers

- `download` fetches a data file from the book's repository **only if it isn't already there**, and raises on HTTP errors.
- `cs` is the **selectors** module: a readable way to choose columns by name, type, or pattern.

In [ ]:
from pathlib import Path
import urllib.request
import warnings

import polars as pl
import polars.selectors as cs
from polars.testing import assert_frame_equal


BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data"
DATA_DIR = Path("sample_data")


def download(relative_path: str) -> Path:
    """Download BASE_URL/relative_path into DATA_DIR (unless it exists) and return the local path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        urllib.request.urlretrieve(f"{BASE_URL}/{relative_path}", temporary)
        temporary.rename(path)
    return path

In [ ]:
print(f"Polars {pl.__version__}")

### Download the data

We use three datasets from the book's repository:

| File | What it contains |
|---|---|
| `fruit.csv` | a tiny table of fruit, handy for seeing what a `group_by` does |
| `top2000-2023.xlsx` | the Dutch *Top 2000* songs of all time (position, title, artist, year) |
| `sales.csv` | ~113,000 bicycle-shop sales (country, product, quantity, profit, revenue, ...) |

In [ ]:
FRUIT_CSV = download("fruit.csv")
TOP2000_XLSX = download("top2000-2023.xlsx")
SALES_CSV = download("sales.csv")

print(FRUIT_CSV, TOP2000_XLSX, SALES_CSV, sep="\n")

## 1. Split, apply, and combine

Almost every aggregation follows the same three steps:

1. **Split** the rows into groups, based on the values of one or more columns.
2. **Apply** an aggregation (sum, mean, count, ...) to each group.
3. **Combine** the per-group results into a new DataFrame, with one row per group.

In Polars, `df.group_by(...)` does step 1, and the aggregation you call next does steps 2 and 3.

## 2. The GroupBy context

`df.group_by("column")` does **not** compute anything yet. It returns a *GroupBy context*: a description of how the DataFrame will be split. Calling an aggregation method on it (such as `.len()`) does the work.

In [ ]:
fruit = pl.read_csv(FRUIT_CSV)
fruit

In [ ]:
fruit_grouped = fruit.group_by("is_round")
print(type(fruit_grouped))

### Counting rows per group

`GroupBy.len()` counts the rows in each group. The order of the groups is **not guaranteed**: it can change between runs and between Polars versions (your output may list `true` and `false` in a different order than the book). When order matters, sort the result.

In [ ]:
(
    fruit_grouped.len()
                 .sort("is_round")
)

### Other GroupBy methods

Chapter 13 lists the methods available on a GroupBy context (Table 13-1): `agg`, `all`, `count`, `first`, `head`, `last`, `len`, `map_groups`, `max`, `mean`, `median`, `min`, `n_unique`, `quantile`, `sum`, `tail`, and iteration over the groups.

Two practical notes for Polars 2.1:

- **`count()` is a leftover.** `len()` is the current name; the cell below prints what your version does with `count()`.
- **Iterating** yields `(key, sub_dataframe)` pairs, where the key is always a **tuple**, even for a single grouping column.

In [ ]:
fruit.group_by("is_round").count()


In [ ]:
for key, group in fruit.group_by("is_round", maintain_order=True):
    print(f"key={key}, group={group}")

`map_groups` hands each group to a Python function as a DataFrame. It is flexible but slow, because Python code runs once per group and Polars cannot optimize it. Prefer expressions in `.agg()` (Section 6) whenever you can. Chapter 17 of the book covers Python functions in depth.

In [ ]:
(
    fruit.group_by("is_round")
         .map_groups(lambda group: group.head(1))
)


## 3. The Top 2000 dataset

The Top 2000 lists the 2,000 best songs of all time, as voted by the Dutch public in 2023.

In [ ]:
top2000 = pl.read_excel(TOP2000_XLSX)


In [ ]:
print(top2000.schema)

In [ ]:
top2000.head()

### Telling Polars the data is sorted (`set_sorted`)

If you know a column is already sorted, `set_sorted` records that fact so Polars can use **fast paths** that are only possible for sorted data.

> **Warning.** Polars does *not* check your claim. If the column is not actually sorted, results can be silently wrong. So: **verify first, then declare.** Below we check with `is_sorted()` before calling `set_sorted`.

In [ ]:
top2000.select(pl.col("positie").is_sorted())

In [ ]:
top2000 = top2000.set_sorted("positie")
top2000.get_column("positie").flags

### Aggregating into a list per group

If you give `.agg()` an expression that does *not* reduce to a single value, Polars collects the values of each group into a **List**. Here we build a "artist - title" string per song, which yields a list of strings per year.

The book spells the separator out with `pl.lit(" - ")`; `pl.concat_str(..., separator=...)` is shorter.

In [ ]:
(
    top2000.group_by("jaar")
    .agg(songs=pl.concat_str("artiest", "titel", separator=" - "))
    .sort("jaar", descending=True)
)

## 4. The descriptives: `head`, `tail`, `first`, `last`

`GroupBy.head(n)` returns the first `n` rows of **each group**, and `GroupBy.tail(n)` the last `n`. Because the Top 2000 is sorted by position, the first rows of each year are that year's best-ranked songs.

We pass `maintain_order=True` so the group rows come out in the original (sorted) order. This is slower than the default, because Polars can no longer process groups in parallel with arbitrary output order, so only use it when order matters.

In [ ]:
(
    top2000.group_by("jaar", maintain_order=True)
    .head(3)
    .sort("jaar", descending=True)
    .head(9)
)

In [ ]:
(
    top2000.group_by("jaar", maintain_order=True)
    .tail(3)
    .sort("jaar", descending=True)
    .head(9)
)

`GroupBy.first()` is the same as `head(1)` and `GroupBy.last()` is the same as `tail(1)`, but they read better and return the first/last **value of every column**.

### An alternative: filter with a window function

The "top 3 per group" result can also be written with `rank().over(...)` and a `filter`.

In [ ]:
by_group_by = (
    top2000.group_by("jaar", maintain_order=True)
    .head(3)
    .sort("jaar", "positie")
)

by_window = (
    top2000.filter(pl.col("positie").rank("ordinal").over("jaar") <= 3)
    .sort("jaar", "positie")
)


In [ ]:
by_group_by

In [ ]:
by_window

### Which artists have the most songs?

Group by artist, count with `len()`, and sort. Many artists tie on the count, and ties are ordered arbitrarily, so we add a **second sort key** (the artist name) to make the output reproducible.

In [ ]:
(
    top2000.group_by("artiest")
    .len()
    .sort(["len", "artiest"], descending=[True, False])
    .head(10)
)

## 5. The sales dataset and the standard aggregations

For the next methods we use a bigger dataset with ~113,000 sales records.

In [ ]:
sales = pl.read_csv(SALES_CSV)

sales

### `max()`: the most expensive sub-categories

We select the columns we need first, group by **two** columns (so each category/sub-category combination is one group), take the maximum of the remaining column, and sort. We break ties on price with the names so the result is reproducible.

In [ ]:
(
    sales.select("Product_Category", "Sub_Category", "Unit_Price")
         .group_by("Product_Category", "Sub_Category")
         .max()
         .sort(["Unit_Price", "Product_Category", "Sub_Category"], descending=[True, False, False])
         .head(10)
)

### `sum()`: total profit per country

In [ ]:
(
    sales.select("Country", "Profit")
         .group_by("Country")
         .sum()
         .sort("Profit", descending=True)
)

### `n_unique()`: how many distinct products per sub-category?

In [ ]:
(
    sales.select("Sub_Category", "Product")
         .group_by("Sub_Category")
         .n_unique()
         .sort(["Product", "Sub_Category"], descending=[True, False])
         .head(10)
)

### `mean()`: average order quantity per age group

In [ ]:
(
    sales.select("Age_Group", "Order_Quantity")
         .group_by("Age_Group")
         .mean()
         .sort("Order_Quantity", descending=True)
)

### `quantile()` and `median()`

`quantile(0.9)` returns the value below which 90% of the group's values fall. `median()` is the same as `quantile(0.5)`.


In [ ]:
(
    sales.select("Age_Group", "Revenue")
         .group_by("Age_Group")
         .quantile(0.9)
         .sort("Revenue", descending=True)
)

### Eager and lazy agree

Everything above also works in the **lazy API**: call `.lazy()`, build the query, and `.collect()`. Lazy queries let Polars optimize the whole plan (for instance, reading only the columns you use). We check the result matches the eager one. We use `check_row_order=False` because group order is not guaranteed.

In [ ]:
eager_result = sales.select("Country", "Profit").group_by("Country").sum()

lazy_result = (
    sales.lazy()
         .group_by("Country")
         .agg(pl.col("Profit").sum())
         .collect()
)


In [ ]:
assert_frame_equal(eager_result, lazy_result, check_row_order=False)

## 6. Advanced methods: `GroupBy.agg()`

The shortcut methods (`.sum()`, `.mean()`, ...) apply **one** function to **all** remaining columns. `GroupBy.agg()` lets you:

- collect values into a list per group
- choose the output column names
- apply several aggregations to several columns at once
- use your own functions that build expressions

### Aggregate values into a list

Passing a bare column to `agg` gives every value of the group, as a List.

In [ ]:
(
    sales.group_by("Country")
         .agg(pl.col("Profit"), pl.col("Revenue"))
         .sort("Country")
)

### Rename aggregated columns

Three ways to name the output columns:

- `Expr.alias("name")`
- the **keyword** form, `name=expression`
- the `name` namespace: `.name.prefix(...)`, `.name.suffix(...)`, `.name.map(...)`

Note that `Cost=pl.col("Revenue") - pl.col("Profit")` is *not* reduced, so it also becomes a list per group. (`sales` already has a `Cost` column; inside `agg` the new name simply applies to the result.)

In [ ]:
(
    sales.group_by("Country")
    .agg(
        pl.col("Profit").alias("All Profits Per Transaction"),
        pl.col("Revenue").name.prefix("All "),
        Cost=pl.col("Revenue") - pl.col("Profit"),
    )
    .sort("Country")
)

### Apply several aggregations at once

Give `agg` a list of expressions. Each expression is evaluated per group. `pl.len()` counts the rows of each group (the expression form of `GroupBy.len()`).

In [ ]:
(
    sales.group_by("Country")
         .agg(
             pl.len().alias("Transactions"),
             pl.col("Profit").sum().name.prefix("Total "),
             pl.col("Profit").mean().alias("Average Profit per Transaction"),
             pl.col("Revenue").sum().name.prefix("Total "),
             pl.col("Revenue").mean().alias("Average Revenue per Transaction"),
         ).sort("Country")
)

### Apply one aggregation to many columns

The book first selects the columns and then uses `pl.all()`. A more explicit alternative is a **selector**: `cs.by_name("Profit", "Revenue")` picks columns by name, and you can also use `cs.numeric()`, `cs.starts_with(...)`, and so on. With selectors you can skip the preliminary `select`, and the code says exactly which columns are summarized.

In [ ]:
by_all = (
    sales.select("Country", "Profit", "Revenue")
         .group_by("Country")
         .agg(
             pl.all().sum().name.prefix("Total "),
             pl.all().mean().name.prefix("Average "),
         ).sort("Country")
)

by_selector = (
    sales.group_by("Country")
         .agg(
             cs.by_name("Profit", "Revenue").sum().name.prefix("Total "),
             cs.by_name("Profit", "Revenue").mean().name.prefix("Average "),
         ).sort("Country")
)

assert_frame_equal(by_all, by_selector)
by_selector

### Counting rows that satisfy a condition

A comparison such as `pl.col("Profit") > 1000` produces a **Boolean** column. Aggregating it as-is gives a list of booleans per group. But `True` counts as 1 and `False` as 0, so `.sum()` **counts the rows for which the condition holds**.

In [ ]:
(
    sales.group_by("Country")
         .agg(
             (pl.col("Profit") > 1000).alias("Profit > 1000"),
             (pl.col("Profit") > 1000).sum().alias("Transactions with Profit > 1000"),
         ).sort("Country")
)

### Custom helper functions that return expressions

Because `agg` accepts expressions, you can wrap a pattern in a **Python function that returns expressions**. This is safe and fast: the Python function runs once, *before* Polars executes, and Polars then runs the resulting expressions in Rust.


In [ ]:
def transactions_above_threshold(column: str, threshold: float) -> list[pl.Expr]:
    """Return two expressions: the raw condition per row, and how many rows exceed the threshold."""
    condition = (pl.col(column) > threshold).alias(f"{column} > {threshold}")
    count = (pl.col(column) > threshold).sum().alias(f"Transactions with {column} > {threshold}")
    return [condition, count]


(
    sales.group_by("Country")
         .agg(transactions_above_threshold("Profit", 999))
         .sort("Country")
)

### Filtering inside an aggregation

You can also restrict *which rows* of each group enter an aggregation, by calling `filter` on the expression. Below, the share of transactions with high profit per country is computed in a single pass.

In [ ]:
(
    sales.group_by("Country")
         .agg(
             pl.len().alias("Transactions"),
             pl.col("Profit").filter(pl.col("Profit") > 1000).sum().alias("Profit from large transactions"),
             (pl.col("Profit") > 1000).mean().alias("Share of large transactions"),
         ).sort("Country")
)

## 7. Row-wise aggregations

So far we aggregated **down** a column. Sometimes you need to aggregate **across** columns within each row. Polars has ready-made horizontal functions: `pl.sum_horizontal`, `pl.mean_horizontal`, `pl.min_horizontal`, `pl.max_horizontal`, `pl.all_horizontal`, `pl.any_horizontal`. For anything else there are two general tools:

- `pl.fold(acc, function, exprs)`: start from an **initial accumulator** `acc` and combine it with each expression in turn.
- `pl.reduce(function, exprs)`: like `fold`, but the **first** expression is the initial accumulator.

The accumulator is updated as `acc = function(acc, next_value)`, from left to right. For `[2, 3, 4]` and `acc = 0`: `0+2=2`, `2+3=5`, `5+4=9`.

In [ ]:
fold_example = pl.DataFrame({"col1": [2], "col2": [3], "col3": [4]})

fold_example.with_columns(
    sum=pl.fold(
        acc=pl.lit(0),
        function=lambda acc, x: acc + x,
        exprs=pl.all(),
    )
)

### Watch the data type of the accumulator

Look at the dtype of the `sum` column above. The book shows `i64`, but in Polars 2.1 you should see **`i32`**. The reason: `pl.lit(0)` is an **Int32** literal (Polars picks the smallest integer type that fits), and `fold` produces the dtype of its accumulator.

That matters. If your inputs are floats or big integers, an `Int32` accumulator can give you a surprising dtype, or overflow. Always give the accumulator an explicit type.

In [ ]:
print("dtype of pl.lit(0):          ", pl.select(pl.lit(0)).schema)
print("dtype of pl.lit(0, Int64):   ", pl.select(pl.lit(0, dtype=pl.Int64)).schema)
print("dtype of pl.lit(0.0):        ", pl.select(pl.lit(0.0)).schema)

In [ ]:
fold_example.with_columns(
    sum_int64=pl.fold(
        acc=pl.lit(0, dtype=pl.Int64),
        function=lambda acc, x: acc + x,
        exprs=pl.all(),
    )
)

### Weighted sum per row

A realistic use of `fold`: multiply each product column by a weight, then add the results up per row. The weights are floats, so the accumulator should be a **float** too (`pl.lit(0.0)`).


In [ ]:
products = pl.DataFrame(
    {
        "product_A": [10, 20, 30],
        "product_B": [20, 30, 40],
        "product_C": [30, 40, 50],
    }
)

weights = {"product_A": 0.5, "product_B": 1.5, "product_C": 2.0}

weighted_exprs = [
    (pl.col(product) * weight).alias(product)
    for product, weight in weights.items()
]

products.with_columns(
    weighted_sum=pl.fold(
        acc=pl.lit(0.0),
        function=lambda acc, x: acc + x,
        exprs=weighted_exprs,
    )
)

### Prefer the built-in horizontal functions

Summing is so common that Polars has `pl.sum_horizontal`. It is vectorized, has no Python lambda in the middle, and picks a sensible output dtype. `pl.reduce` gives the same result when the first column is the starting value. We check that all three agree.

In [ ]:
via_fold = products.select(
    pl.fold(pl.lit(0.0), lambda acc, x: acc + x, weighted_exprs).alias("weighted_sum")
)
via_sum_horizontal = products.select(pl.sum_horizontal(weighted_exprs).alias("weighted_sum"))
via_reduce = products.select(pl.reduce(lambda acc, x: acc + x, weighted_exprs).alias("weighted_sum"))


In [ ]:
via_fold

In [ ]:
via_sum_horizontal

In [ ]:
via_reduce

Use `fold`/`reduce` when the combining step is something for which Polars has no horizontal function (for example, a computing a running product or a custom score).

## 8. Window functions in the select context: `Expr.over()`

`group_by().agg()` **collapses** each group to one row. Often you want the opposite: keep every row, and *add* a column that holds a group-level result. That is what `Expr.over()` does: it computes an expression per group and maps the result back to the original rows.

Arguments of `over` (Table 13-3 in the book):

| Argument | Meaning |
|---|---|
| `partition_by` | the column(s) that define the groups |
| `order_by` | order the rows within each group before computing the expression |
| `mapping_strategy` | `"group_to_rows"` (default): result per row; `"join"`: the whole group's result as a List on every row; `"explode"`: one row per result value (changes the number of rows) |

### Rank songs within their release year

In [ ]:
(
    top2000.select(
        "jaar",
        "artiest",
        "titel",
        "positie",
        year_rank=pl.col("positie").rank().over("jaar"),
    ).sample(10, seed=42)
)

`rank()` returns floats by default because ties get the *average* rank. Use `rank("ordinal")` for unique integer ranks.

The same rank, sorted, for 2013 only:

In [ ]:
(
    top2000.filter(pl.col("jaar") == 2013)
           .select("jaar", "artiest", "titel", "positie", year_rank=pl.col("positie").rank("ordinal").over("jaar"))
           .sort("year_rank")
           .head(5)
)

### `over` versus `group_by` + `join`

You could get the same result by aggregating with `group_by` and then joining the result back. `over` does it in one step. We verify the two are equivalent for "best position in each song's year". Positions are unique, so sorting by `positie` makes both frames comparable.

In [ ]:
with_over = top2000.with_columns(best_in_year=pl.col("positie").min().over("jaar")).sort("positie")

best_per_year = top2000.group_by("jaar").agg(best_in_year=pl.col("positie").min())
with_join = top2000.join(best_per_year, on="jaar", how="left", validate="m:1").sort("positie")


In [ ]:
assert_frame_equal(with_over, with_join)

### A share of the group total

Another very common use of `over`: divide each row by its group total, without a join.

In [ ]:
(
    sales.select(
        "Country",
        "Product_Category",
        "Profit",
        profit_share_of_country=pl.col("Profit") / pl.col("Profit").sum().over("Country"),
    )
    .head(5)
)

### `mapping_strategy`

With the default, a result of the same length as the group is placed on the matching rows. With `"join"`, every row receives the group's **entire result as a list**. Both are shown here on a tiny table so you can see the difference.

In [ ]:
small = pl.DataFrame({"group": ["a", "a", "b", "b", "b"], "value": [1, 2, 3, 4, 5]})

small.with_columns(
    default=pl.col("value").cum_sum().over("group"),
    joined=pl.col("value").cum_sum().over("group", mapping_strategy="join"),
)

## 9. Dynamic grouping: `group_by_dynamic`

With temporal data it is often useful to group by **time windows**: per week, per month, per quarter. `df.group_by_dynamic()` creates windows of a fixed step and length and assigns rows to them. Unlike a normal `group_by`, **a row can fall into more than one window** (when windows overlap) or into none (when there are gaps).

| Argument | Meaning |
|---|---|
| `index_column` | the temporal (or integer) column to window over. It must be **sorted**. |
| `every` | the **step**: how far apart the window starts are |
| `period` | the **length** of each window (defaults to `every`) |
| `offset` | shifts the window starts (e.g. `every="1d", offset="9h"` starts windows at 9 a.m.) |
| `closed` | which window edges are inclusive: `"left"` (default), `"right"`, `"both"`, `"none"` |
| `label` | which edge labels the window: `"left"` (default), `"right"`, `"datapoint"` |
| `start_by` | how the first window is placed, e.g. `"window"` (default), `"datapoint"`, `"monday"` |
| `group_by` | group first, then window within each group |

### Duration strings

`every`, `period`, and `offset` take strings made of a number and a unit. You can combine them: `"1y6mo1w5d"` is one year, six months, one week, and five days.

| String | Meaning | String | Meaning |
|---|---|---|---|
| `1ns` | 1 nanosecond | `1d` | 1 calendar day |
| `1us` | 1 microsecond | `1w` | 1 calendar week |
| `1ms` | 1 millisecond | `1mo` | 1 calendar month |
| `1s` | 1 second | `1q` | 1 calendar quarter |
| `1m` | 1 minute | `1y` | 1 calendar year |
| `1h` | 1 hour | `1i` | 1 index count (integer index columns) |

### The three window shapes

- **Non-overlapping:** `every="1d", period="1d"` (windows touch end to end)
- **Overlapping:** `every="1d", period="2d"` (each window is longer than the step)
- **Gapped:** `every="2d", period="1d"` (windows are shorter than the step; some rows belong to no window)

### A small dataset

We create two weeks of daily-ish sales for two stores, open on weekdays only. (This is the dataset the book uses for rolling and upsampling too.) `dt.weekday()` is 1 for Monday through 7 for Sunday, so `< 6` keeps Monday to Friday.

We build the table in **date order** like the book, and then make a second copy sorted by `store` and then `date`. Window functions that group (`group_by_dynamic`, `rolling`, `upsample` with `group_by`) need the time column sorted *within each group*, and sorting explicitly is safer than *claiming* sortedness with `set_sorted`.

In [ ]:
dates = pl.date_range(
    start=pl.date(2024, 4, 1),
    end=pl.date(2024, 4, 26),
    interval="2d",
    eager=True,
)
dates = dates.filter(dates.dt.weekday() < 6)
dates_repeated = pl.concat([dates, dates]).sort()

small_sales_df = pl.DataFrame(
    {
        "date": dates_repeated,
        "store": ["Store A", "Store B"] * dates.len(),
        "sales": [
            200, 150, 220, 160, 250, 180, 270, 190, 280, 210,
            210, 170, 220, 180, 240, 190, 250, 200, 260, 210,
        ],
    }
)

# Sorted by store, then by date within each store.
sales_by_store = small_sales_df.sort("store", "date")
sales_by_store

### Non-overlapping weekly windows, per store

`every="1w"` makes one window per calendar week. We turn on `include_boundaries=True` so you can see each window's `_lower_boundary` and `_upper_boundary`. Windows are `closed="left"` by default: `[lower, upper)`.

In [ ]:
(
    sales_by_store.group_by_dynamic(
        "date",
        every="1w",
        group_by="store",
        include_boundaries=True
    ).agg(
        pl.col("sales").sum().alias("weekly_sales"),
        pl.len().alias("n_days")
    )
)

### Overlapping windows

With `period="2w"` each window spans two weeks but a new one starts every week, so most rows are counted in **two windows**.

In [ ]:
(
    sales_by_store.group_by_dynamic(
        "date", every="1w", period="2w", group_by="store", include_boundaries=True
    )
    .agg(pl.col("sales").sum().alias("two_week_sales"), pl.len().alias("n_days"))
)

### Gapped windows

With `period="3d"` each window covers only Monday to Wednesday of its week. Rows on Thursday and Friday belong to **no window** and are dropped.

In [ ]:
(
    sales_by_store.group_by_dynamic(
        "date", every="1w", period="3d", group_by="store", include_boundaries=True
    )
    .agg(pl.col("sales").sum().alias("early_week_sales"), pl.len().alias("n_days"))
)

### The `closed` argument

`closed` controls whether a value that sits exactly **on** a window edge is included. In interval notation (square bracket = inclusive, round bracket = exclusive):

| `closed` | Interval | Includes lower edge | Includes upper edge |
|---|---|---|---|
| `"left"` | `[a, b)` | yes | no |
| `"right"` | `(a, b]` | no | yes |
| `"both"` | `[a, b]` | yes | yes |
| `"none"` | `(a, b)` | no | no |

(The book's Table 13-6 prints the interval for `right` and `none` inconsistently; the table above is the standard notation.)

The loop below groups six consecutive days into windows of two days, and lists which values land in each window for each setting.

In [ ]:
closed_demo = pl.DataFrame(
    {
        "day": pl.date_range(pl.date(2024, 1, 1), pl.date(2024, 1, 6), interval="1d", eager=True),
        "v": [1, 2, 3, 4, 5, 6],
    }
)

for closed in ["left", "right", "both", "none"]:
    print(f"closed={closed!r}")
    print(
        closed_demo.group_by_dynamic("day", every="2d", closed=closed, include_boundaries=True)
        .agg(pl.col("v"))
    )

### Real data: monthly profit

On the sales data we can group by month. The `Date` column is read as text, so we parse it, **sort** (dynamic grouping needs a sorted index), and aggregate per month. This cell is wrapped in `try_it`: if the date format in your copy of the file isn't recognized, you'll see the error instead of a stopped notebook (inspect `Date` in the first cell and pass `format=...` to `str.to_date`).

In [ ]:
sales.select("Date", "Day", "Month", "Year").head(3)

In [ ]:
def monthly_profit():
    return (
        sales.select(pl.col("Date").str.to_date(), "Profit")
        .sort("Date")
        .group_by_dynamic("Date", every="1mo")
        .agg(pl.col("Profit").sum())
        .head(6)
    )


try_it(monthly_profit)

## 10. Rolling aggregations: `rolling`

`group_by_dynamic` uses windows of fixed size and step. `df.rolling()` instead creates **one window per row**, anchored at that row's value and looking back by `period`. It is ideal for moving averages and running sums.

| Argument | Meaning |
|---|---|
| `index_column` | the column that anchors each window (must be sorted, within groups if you use `group_by`) |
| `period` | the size of the window |
| `offset` | shifts the window backward or forward |
| `closed` | which edges are inclusive (default `"right"`: the window is `(t - period, t]`) |
| `group_by` | compute the rolling windows within each group |

### Rolling 7-day sales per store

For every row, the window covers the current date and the six days before it. Early rows have fewer than seven days available, so their sum covers only what exists.

In [ ]:
result = sales_by_store.rolling(
    index_column="date",
    period="7d",
    group_by="store",
).agg(sum_of_last_7_days_sales=pl.sum("sales"))

# validate="1:1" raises if (date, store) is not unique on either side.
final_df = sales_by_store.join(result, on=["date", "store"], how="left", validate="1:1").sort("store", "date")
final_df

### A shorter way: `rolling_sum_by`

Joining the `rolling` result back onto the original table is a very common pattern, and Polars has a shortcut for it: the `rolling_*_by` expressions (`rolling_sum_by`, `rolling_mean_by`, `rolling_min_by`, ...). They compute the rolling statistic **inside** `with_columns`, keeping every row, so there is no join. Combine with `.over("store")` to do it per group.


In [ ]:
rolling_by = sales_by_store.with_columns(
    sum_of_last_7_days_sales=pl.col("sales").rolling_sum_by("date", window_size="7d").over("store")
)

rolling_by

A rolling **mean** works the same way, and `closed` still decides whether the window's edges are included:

In [ ]:
sales_by_store.with_columns(
    mean_last_7_days=pl.col("sales").rolling_mean_by("date", window_size="7d").over("store"),
    mean_last_7_days_strict=pl.col("sales").rolling_mean_by("date", window_size="7d", closed="left").over("store"),
)

## 11. Upsampling: `upsample`

The opposite of aggregating temporal data is **upsampling**: if the resolution is too coarse, `df.upsample()` inserts the missing time steps. The new rows contain nulls, which you then fill (Chapter 8 of the book covers filling).

| Argument | Meaning |
|---|---|
| `time_column` | the temporal column (must be sorted for the output to make sense) |
| `every` | the new step |
| `group_by` | upsample separately within each group |
| `maintain_order` | keep the order predictable (slower) |

Our table only has a row every other day (weekdays only). Let's upsample it to **daily**, per store.

> **Why we sort first.** `upsample` relies on the data being ordered by group and by time. We sort by `store` and `date` instead of *claiming* it with `set_sorted`.

In [ ]:
upsampled_small_sales_df = sales_by_store.upsample(
    time_column="date",
    every="1d",
    group_by="store",
    maintain_order=True
)

upsampled_small_sales_df


### Fill the gaps

The new rows have nulls. Two steps:

1. **Forward-fill** the `store` column (carry the last store name down). Depending on the version, `upsample` may already fill the group column, and this step is harmless either way.
2. **Interpolate** `sales` linearly between known values. The book interpolates over the whole column, which would also blend the last day of Store A into the first day of Store B. We use `.over("store")` so each store is interpolated **on its own**.

In [ ]:
(
    upsampled_small_sales_df.with_columns(
        pl.col("store").forward_fill()
    ).with_columns(
        pl.col("sales").interpolate().over("store")
    )
)

## Takeaways

- `df.group_by(...)` only **describes** the split; calling an aggregation (`len`, `sum`, `mean`, `quantile`, `agg`, ...) computes it. Group order is **not deterministic**, so sort (on enough columns to break ties) when order matters.
- `GroupBy.agg()` accepts any expressions: lists per group, custom names (`alias`, keyword form, `name.prefix`), many columns at once (`pl.all()` or **selectors**), counted conditions (`(cond).sum()`), and functions that return expressions.
- `GroupBy.head/tail/first/last` keep rows; `maintain_order=True` preserves the original order but is slower. `count()` is superseded by `len()`.
- For row-wise aggregation prefer `pl.sum_horizontal` and friends. With `pl.fold`, **give the accumulator an explicit dtype**: the output dtype follows the accumulator (`pl.lit(0)` is Int32).
- `Expr.over()` adds group-level results **without** collapsing rows; it replaces the `group_by` + `join` pattern.
- `group_by_dynamic` builds non-overlapping, overlapping, or gapped time windows (`every`, `period`, `offset`, `closed`); `rolling` builds one window per row, and `rolling_*_by(...).over(...)` is the shorter form.
- `upsample` adds missing time steps; fill the new nulls afterwards, and fill/interpolate **within each group**.
- `set_sorted` can unlock fast paths but is **never verified**. Check with `is_sorted()` or sort explicitly; a false claim can silently give wrong results.

## Next up

Chapter 14 shows how to combine multiple DataFrames with **joins** and **unions**.

## Try it yourself


1. **Group sizes.** In `top2000`, count songs per `jaar` and list the five years with the most songs. Break ties by year so the answer is reproducible.


2. **Several statistics.** For each `Product_Category` in `sales`, compute the number of transactions, the total `Revenue`, and the median `Profit` in a single `agg`. Name the columns yourself.

3. **Selectors.** Use `cs.numeric()` to compute the mean of **every numeric column** of `sales`, per `Country`. Which columns are included? Is `Year` really something you want to average?

4. **Fold dtype.** Change the weights in the weighted-sum example to `0.25`, `1.25`, and `2.75`. Compute the result with `pl.fold(pl.lit(0), ...)` and with `pl.fold(pl.lit(0.0), ...)`. Compare the dtypes and values, and then compute it with `pl.sum_horizontal`.


5. **Window function.** Add a column to `sales` with each transaction's `Profit` as a **share of its `Product_Category`'s total profit**, using `over`. Verify that the shares of each category sum to 1.

6. **Time windows.** Using `sales_by_store`, compute the sales per store in **overlapping** windows with `every="2d", period="6d"`. How many windows contain each date?